# The book made faster

Section 1.2 of the lecture notes, continued.  The fold is written once, clearly, and we now
write it four more times and measure whether any of them is worth having.

The order matters.  The clear implementation is the one a reader understands and the one every
other is checked against; an optimisation is a second implementation *shown* to beat a first,
on a stream, in a regime named out loud.  Every number below is measured in this notebook, on
this machine, now.  None is quoted from anywhere.

In [1]:
import timeit

import pandas as pd

from unito26.lob import benchmark, config
from unito26.lob.messages import BUY, SELL, GridDepth, ReportedDepth, SweepSize
from unito26.lob.orderbook import (
    AXIS_B_VARIANTS, AggregateBook, BitmapBook, CachedBestBook, HeapBook, TickArrayBook,
)
from unito26.lob.session import run
from unito26.lob.simulate import MarkParams
from unito26.lob.statistics import SessionStatistics
from unito26.lob.visualization import band_width_figure, use_template
from unito26.lob.worked_examples import CATALOGUE, check, reflect

use_template()

HORIZON = 1800.0
REPEAT = 3
SPEC = SessionStatistics((GridDepth(1),), (SweepSize(100),), (1,))

shallow = benchmark.session("shallow", config.shallow_mark_params(), HORIZON, seed=11)
deep = benchmark.session("deep", config.deep_mark_params(), HORIZON, seed=11)

for stream in (shallow, deep):
    print(f"{stream.name:<8} {len(stream.messages):>7,} messages, "
          f"L = {stream.occupied_levels():>4} occupied levels at the end")

shallow   53,610 messages, L =   32 occupied levels at the end
deep      53,285 messages, L =  376 occupied levels at the end


Two streams, from the same order flow and two different mark distributions: one where limit
orders cluster within a few ticks of the touch, and one where they spread out.  They differ in
nothing else, and $L$ — the number of occupied levels — is what they differ *in*.

A stream is materialised before anything is timed.  Every variant then folds the identical list
of messages, so a difference in the timings has one cause.

## 1. Matching is a rounding error; the fold is what must be timed

Four measurements of the same stream, each adding one layer.

In [2]:
layers = {
    "apply, recording nothing": benchmark.time_apply(AggregateBook, shallow, False, REPEAT),
    "apply, recording fills": benchmark.time_apply(AggregateBook, shallow, True, REPEAT),
    "fold to a session, no statistics": benchmark.time_fold(
        AggregateBook, shallow, ReportedDepth(5), SPEC, False, REPEAT),
    "fold to a session, statistics on": benchmark.time_fold(
        AggregateBook, shallow, ReportedDepth(5), SPEC, True, REPEAT),
}
whole = layers["fold to a session, statistics on"]
for name, seconds in layers.items():
    print(f"{name:<34} {seconds:7.3f} s   {100 * seconds / whole:5.1f}% of the whole")

apply, recording nothing             0.033 s     2.2% of the whole
apply, recording fills               0.090 s     6.0% of the whole
fold to a session, no statistics     0.396 s    26.2% of the whole
fold to a session, statistics on     1.516 s   100.0% of the whole


The matching itself is a small share of a session's cost.  Recording the fills nearly triples
it, writing a row per message multiplies it again, and the statistics are the largest term of
the four.

That bounds the ladder before a single rung is built.  Every variant below changes only how the
best price is found, and the best price is found inside `apply`.  Whatever the ladder wins, it
wins a fraction of that first line — so an implementation that made matching free would leave
the fold above almost where it is.

This is the calculation to do before the optimisation, and it is the one usually done after.

## 2. Cumulative time, not own time

Inside the matching, the picture is different.  `best_price` is read by every incoming order and
scans the occupied prices, so its share is a function of $L$.

In [3]:
for stream in (shallow, deep):
    share = benchmark.best_price_share(AggregateBook, stream)
    print(f"{stream.name:<8} L = {stream.occupied_levels():>4}   "
          f"best_price is {share:6.1%} of run time")

shallow  L =   32   best_price is  15.1% of run time


deep     L =  376   best_price is  32.5% of run time


The measurement has a trap in it.  `best_price` finds its answer by calling `max`, and a
profiler bills a builtin to itself: the method's *own* time is nearly nothing, while the work it
is responsible for sits one frame below it.  `best_price_share` sums the **cumulative** column
for that reason.  Reading the own-time column instead reports about three per cent where the
truth is ten times that, and three per cent is the number at which an optimisation is abandoned.

A profile is not self-explanatory, and which column answers the question is part of the
question.

## 3. The ladder varies one thing

Five books.  The matching logic is written once, in `AggregateBook`, and inherited unchanged;
each variant overrides the best-price lookup and the write that keeps its index in step.

In [4]:
PRIMITIVES = ("best_price", "set_size", "levels_map", "for_prices", "size_at")

print(f"{'AggregateBook':<16} defines all of them: it is the baseline")
for cls in AXIS_B_VARIANTS[1:]:
    overridden = [name for name in PRIMITIVES if name in cls.__dict__]
    print(f"{cls.__name__:<16} overrides {', '.join(overridden)}")

AggregateBook    defines all of them: it is the baseline
CachedBestBook   overrides best_price, set_size
HeapBook         overrides best_price, set_size
BitmapBook       overrides best_price, set_size, for_prices
TickArrayBook    overrides best_price, set_size, levels_map, for_prices, size_at


The first three keep the two dicts as storage and put an index beside them.  The last stops
varying one factor on purpose — it fuses storage and index — and that is stated rather than
hidden, so the step to it measures the fusion and not something else.

## 4. Caching the best price, and the three cases of invalidation

The best price is read far more often than it changes, so remember it.  Invalidation is the
whole of the difficulty, and it has exactly three cases: a level that betters the cache, a level
that empties the cached price, and everything else.

The cache is derived state — recoverable from the levels at any moment — which is what makes it
checkable.

In [5]:
cached = CachedBestBook()
run(cached, deep.messages)
cached.check_cache_is_consistent()

print("cache agrees with a full rescan after", f"{len(deep.messages):,}", "messages")
print("best bid / best ask:", cached.best_bid_price, "/", cached.best_ask_price)

cache agrees with a full rescan after 53,285 messages
best bid / best ask: 9999 / 10004


A cache whose consistency cannot be asserted is a cache that will be wrong quietly.  The check
costs a full rescan, which is exactly what the cache exists to avoid — so it runs in the tests
and in a debug run, and never on the hot path.

## 5. A heap, and what lazy deletion costs

`heapq` cannot remove an entry from the middle, so a price whose level empties is left in the
heap and discarded when it reaches the top.  The heap therefore grows with every level ever
created, rather than with the levels that exist.

In [6]:
heaped = HeapBook()
run(heaped, deep.messages)

live = {BUY: len(heaped.levels_map(BUY)), SELL: len(heaped.levels_map(SELL))}
stale = heaped.heap_overhead()
for direction, name in ((BUY, "bid"), (SELL, "ask")):
    print(f"{name}: {live[direction]:>4} live levels, {stale[direction]:>6,} stale entries")

heaped.compact()
print("after compact:", heaped.heap_overhead())

bid:  179 live levels,  8,567 stale entries
ask:  197 live levels, 14,516 stale entries
after compact: {1: 0, -1: 0}


Tens of times as many stale entries as live levels, on half an hour of flow.  The memory
is one cost; the other is that every stale entry at the top is popped before an answer is
returned, so the read the heap was built to make cheap is occasionally linear in the rubbish
above it.

`compact` rebuilds both heaps from the live levels.  Needing it is part of the price of the
technique, and a variant whose docstring does not say when to call it has moved the problem
rather than solved it.

## 6. The occupancy as one integer — and the two sides are not equally cheap

A bitset is how a low-latency book finds the next active level: in C, a hierarchy of 64-bit
words and a count-trailing-zeros instruction.  Python's integers are arbitrary-precision, so the
whole band is *one* integer and each search is one expression:

- the highest set bit is `bits.bit_length() - 1`, which is $O(1)$;
- the lowest set bit is `(bits & -bits).bit_length() - 1`, which is $O(\text{span})$.

The bid side wants the highest price and the ask side the lowest, so the two sides of one bitmap
do not cost the same.

In [7]:
bitmap = BitmapBook.for_prices(deep.prices)
run(bitmap, deep.messages)

CALLS = 20_000
for direction, name in ((BUY, "bid: highest set bit"), (SELL, "ask: lowest set bit")):
    seconds = min(timeit.repeat(
        lambda d=direction: bitmap.best_price(d), number=CALLS, repeat=5))
    print(f"BitmapBook  {name:<22} {1e9 * seconds / CALLS:6.0f} ns per lookup")

BitmapBook  bid: highest set bit       95 ns per lookup
BitmapBook  ask: lowest set bit       185 ns per lookup


The asymmetry is the algorithm's, not the market's: both sides hold a comparable number of
levels, and the ask side pays for the span of the band at every lookup.

## 7. Fusing storage and index

Prices already live on an integer grid, so the tick can *be* the array subscript and the size
read where the occupancy bit is set.  There is no dict at all.

That buys one indirection instead of two.  It costs a band, fixed in advance, occupied or not —
a lookup table costs the whole table — and it buys back the asymmetry of section 6: with an
upper edge to count down from, the ask side is indexed as `ceiling - price`, so the best price
on *either* side is the highest set bit.

In [8]:
array = TickArrayBook.for_prices(deep.prices)
run(array, deep.messages)

print("band:", array.width, "ticks from", array.origin, "to", array.ceiling)
for direction, name in ((BUY, "bid"), (SELL, "ask")):
    seconds = min(timeit.repeat(
        lambda d=direction: array.best_price(d), number=CALLS, repeat=5))
    print(f"TickArrayBook  {name:<4} {1e9 * seconds / CALLS:6.0f} ns per lookup")

band: 1083 ticks from 9471 to 10553


TickArrayBook  bid     100 ns per lookup
TickArrayBook  ask      96 ns per lookup


In [9]:
outside = array.ceiling + 1
try:
    array.set_size(BUY, outside, 100)
except ValueError as error:
    print("write outside the band:", error)

print("read outside the band  :", array.size_at(BUY, outside))

write outside the band: price 10554 is outside the band [9471, 10554); a real book would shift the band or fall back to a sorted map
read outside the band  : 0


A write outside the band raises; a read returns zero.  The asymmetry is deliberate.  Reads run
off the edge in ordinary use — asking for ten grid positions above the best ask walks past the
band whether or not anything is there — and a grid position outside the band genuinely holds
nothing.  A write outside it is a book that cannot represent the market it is in, and the
message says what a real venue does instead.

## 8. They all agree, before any timing is believed

A faster wrong answer is not a result.  The catalogue of section 1.1 has one transition per
branch of the update rule; every variant runs every one of them, and every mirror image too.

In [10]:
checks = 0
for cls in AXIS_B_VARIANTS:
    for example in CATALOGUE:
        check(cls, example)
        check(cls, reflect(example, centre=1000))
        checks += 2

print(f"{checks} checks passed: {len(AXIS_B_VARIANTS)} books x "
      f"{len(CATALOGUE)} transitions x 2 sides")

110 checks passed: 5 books x 11 transitions x 2 sides


The mirrored half is generated rather than written.  Reflecting prices about a centre and
flipping every direction is a symmetry of the matching rule, so the sell-side examples follow
from the buy-side ones — and their passing is itself a test of the claim that the single
expression $\pi d \leq p d$ collapses both sides into one.

The one part of the encoding that is not symmetric is the market-order sentinel, which is $0$
and `sys.maxsize` rather than $\mp\infty$: a mirrored market sell would reflect into an ordinary
marketable limit buy and quietly test something else.  `reflect` mirrors a market order by its
meaning instead of by its price field.

## 9. Measured, in two regimes

In [11]:
timings = pd.DataFrame({
    stream.name: benchmark.time_variants(AXIS_B_VARIANTS, stream, REPEAT)
    for stream in (shallow, deep)
})
for column in list(timings.columns):
    timings[f"{column} speedup"] = timings[column].iloc[0] / timings[column]

print(timings.round(4).to_string())

                shallow    deep  shallow speedup  deep speedup
AggregateBook    0.0530  0.1070           1.0000        1.0000
CachedBestBook   0.0542  0.0641           0.9779        1.6683
HeapBook         0.0649  0.0663           0.8162        1.6127
BitmapBook       0.0611  0.0651           0.8666        1.6424
TickArrayBook    0.0567  0.0604           0.9351        1.7719


On the shallow book every rung is at best a wash and some are a loss: the index costs a write on
every level change, the scan it replaces is over thirty-odd keys, and the write is not repaid.
On the deep book every rung wins, and the fused one wins most.

Memory ranks them differently again.

In [12]:
memory = pd.DataFrame(benchmark.measure_memory(AXIS_B_VARIANTS, deep)).T
print((memory / 1024).round(1).rename(columns=lambda c: f"{c} (KiB)").to_string())

                resident (KiB)  traced_current (KiB)  traced_peak (KiB)
AggregateBook             50.6                  38.2              385.8
CachedBestBook            50.9                  37.6              385.9
HeapBook                 697.0                 511.8              514.1
BitmapBook                51.3                  37.9              771.3
TickArrayBook             22.2                  18.9              771.3


Two measurements because neither alone is enough.  The static walk misses what was allocated and
released along the way; the traced peak misses nothing and attributes everything to the book,
including the stream it was folding.

The heap is the outlier, for the reason section 5 gave.  The fused book is the smallest resident
of all: no dict, no keys, no boxed integers for the prices — a band of machine-sized slots, most
of them zero.

## 10. The conditional

The index is worth what the scan costs, and the scan costs what the book is deep.  So sweep the
depth of the book and read where the two lines cross.

In [13]:
sweep = []
for decay in (0.45, 0.20, 0.08, 0.04, 0.02):
    marks = MarkParams.from_records([
        {"DepthDecay": decay, "MeanLogSize": 4.0, "SigmaLogSize": 0.8, "Lot": 10},
    ])
    stream = benchmark.session(f"decay {decay}", marks, 900.0, seed=11)
    measured = benchmark.time_variants((AggregateBook, TickArrayBook), stream, REPEAT)
    sweep.append({
        "DepthDecay": decay,
        "L": stream.occupied_levels(),
        "AggregateBook": measured["AggregateBook"],
        "TickArrayBook": measured["TickArrayBook"],
        "Speedup": measured["AggregateBook"] / measured["TickArrayBook"],
    })

sweep = pd.DataFrame(sweep).set_index("L")
print(sweep.round(4).to_string())

     DepthDecay  AggregateBook  TickArrayBook  Speedup
L                                                     
33         0.45         0.0256         0.0272   0.9397
35         0.20         0.0266         0.0274   0.9696
87         0.08         0.0306         0.0278   1.1008
164        0.04         0.0376         0.0287   1.3104
326        0.02         0.0479         0.0289   1.6557


In [14]:
band_width_figure(sweep[["AggregateBook", "TickArrayBook"]],
                  "folding one stream: dict scan against tick array, by occupied levels")

The baseline's time grows with $L$ and the array's does not, so the speedup crosses one
somewhere in the middle of the range and the crossing is the whole answer.  A speedup quoted
without its $L$ says nothing.

Whether the bottleneck exists at all is therefore a property of the market rather than of the
code.  A large-tick instrument, where everything rests within a few ticks of the touch, lives at
the left of that table and is made slower by every rung of this ladder.

One more thing the band is exposed to, and it is a good example of a measurement that fails
silently.  A market order carries a sentinel price — zero for a sell — and the sentinel is an
ordinary integer.  Sizing the band from the prices of a stream that contains one gives a band
running from zero to the market: the array still answers every question correctly, and the
fastest rung measures as the slowest.  `for_prices` passes its argument through `grid_prices`
for that reason, and the timing is read against a baseline, which is what makes a result of that
shape visible at all.

## What to take away

Measure before optimising, and measure the thing the optimisation targets.  Matching is a small
share of a session's cost, and a book made infinitely fast leaves most of that cost where it is.

A profile is not self-explanatory.  Own time and cumulative time answer different questions, and
the wrong one here reports three per cent where the truth is thirty.

Correctness first, and by construction: every variant runs the same catalogue and the same
mirror images before a single timing is read.

The answer is conditional, and the condition is the market.  Report the speedup with the regime
it was measured in, or do not report it.